In [2]:

import asyncio
import threading

class SelectorLoopRunner:
    def __init__(self):
        self.ready = threading.Event()
        self.thread = threading.Thread(
            target=self._start,
            daemon=True
        )
        self.thread.start()
        self.ready.wait()

    def _start(self):
        self.loop = asyncio.SelectorEventLoop()
        asyncio.set_event_loop(self.loop)
        self.ready.set()

        try:
            self.loop.run_forever()
        finally:
            self.loop.close()

    def run(self, coro):
        future = asyncio.run_coroutine_threadsafe(
            coro, self.loop
        )
        return future.result()

    def close(self):
        self.loop.call_soon_threadsafe(self.loop.stop)
        self.thread.join()

runner = SelectorLoopRunner()


In [3]:

from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from app.core.config import settings
from dotenv import load_dotenv

load_dotenv()

async def init_checkpointer():
    print("事件循环:", type(asyncio.get_running_loop()).__name__)

    pool = AsyncConnectionPool(
        conninfo=settings.db.checkpoint_url,
        min_size=1,
        max_size=5,
        kwargs={
            "autocommit": True,
            "prepare_threshold": None,
            "row_factory": dict_row,
        },
        open=False,
    )

    try:
        await pool.open()
        await pool.wait(timeout=15)

        checkpointer = AsyncPostgresSaver(pool)
        await checkpointer.setup()

        return pool, checkpointer
    except BaseException:
        await pool.close()
        raise

# 在独立的 SelectorEventLoop 中运行
checkpoint_pool, checkpointer = runner.run(
    init_checkpointer()
)

print("Checkpointer 初始化成功！")


事件循环: _WindowsSelectorEventLoop
Checkpointer 初始化成功！


In [4]:

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

model = init_chat_model(
    "deepseek-v4-flash"
)

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer
)

config = {
    "configurable": {
        "thread_id": "user_001"
    }
}

result = runner.run(
    agent.ainvoke(
        {
            "messages": [
                HumanMessage(content="你好，我叫小明")
            ]
        },
        config=config
    )
)

print(result["messages"][-1].content)


你好，小明！很高兴认识你。有什么我可以帮你的吗？
